# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIRˆ2 dataset using the `mlcroissant` library, referencing all dataset entities via their `@id` fields for full reproducibility and transparency.

### Dataset Source
The dataset is described by a Croissant schema at the following URL:
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Let's load the dataset metadata using `mlcroissant`. This retrieves both schema and descriptive information, providing an overview before data extraction.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
CROISSANT_URL = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(CROISSANT_URL)

# Display dataset metadata using properties
print(f"Name: {dataset.metadata.name}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Version: {dataset.metadata.version}")
print(f"Description: {dataset.metadata.description}")
print(f"Date Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Let's inspect the record sets, fields, and columns available, using their `@id` fields. In Croissant, *record sets* reference logical groups of tabular records, and each column/field is uniquely identified by an `@id`. All entities are referenced by their `@id` below.

In [ ]:
# List all record sets and their @id
record_sets = list(dataset.record_sets())
for rs in record_sets:
    print(f"Record set: @id={rs['@id']}, Name: {rs.get('name','unnamed')}")
    # List fields in each record set
    if 'field' in rs and rs['field']:
        if isinstance(rs['field'], list):
            for field in rs['field']:
                print(f"  Field: @id={field['@id']} Name: {field.get('name','unnamed')}")
        else:
            print(f"  Field: @id={rs['field']['@id']} Name: {rs['field'].get('name','unnamed')}")
    if 'column' in rs and rs['column']:
        if isinstance(rs['column'], list):
            for col in rs['column']:
                print(f"  Column: @id={col['@id']} Name: {col.get('name','unnamed')}")
        else:
            print(f"  Column: @id={rs['column']['@id']} Name: {rs['column'].get('name','unnamed')}")

# Choose a record set for previewing its records (replace with your @id from above, or use the first available if only one)
if record_sets:
    example_record_set_id = record_sets[0]['@id']
    print("\nPreviewing some records from record set:", example_record_set_id)
    # Print first 3 records (dicts)
    for i, rec in enumerate(dataset.records(record_set=example_record_set_id)):
        print(rec)
        if i >= 2:
            break

## 3. Data Extraction
Now, let's load records from each record set into Pandas DataFrames for analysis. Remember to use each record set's `@id`. We'll collect all tables in a dictionary by their `@id`.

In [ ]:
# Extract all record sets by @id
all_record_set_ids = [rs['@id'] for rs in record_sets]

dfs = {}

# Load all dataframes using croissant record_set @id
for rs_id in all_record_set_ids:
    rows = list(dataset.records(record_set=rs_id))
    if rows:  # Only add if records exist
        dfs[rs_id] = pd.DataFrame(rows)

# List dataframes (by record set @id)
print('Loaded DataFrames for these record sets:')
for k in dfs:
    print('-', k, 'shape:', dfs[k].shape)

# Preview columns from the primary record set (replace as needed)
if dfs:
    primary_rs_id = list(dfs.keys())[0]  # Select first loaded record set @id
    print(f"\nColumns in DataFrame for record set '{primary_rs_id}':")
    print(dfs[primary_rs_id].columns.tolist())
    print("\nPreview:")
    display(dfs[primary_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's apply some initial explorations:

- Filtering: Select records by a numeric field (@id).
- Normalization: Compute normalized values for a numeric column (@id).
- Grouping: Aggregate by a key field (@id).

You may replace the `numeric_field_id` and `group_field_id` variables below with actual @id values found above.

In [ ]:
# --- Set your chosen record set and field @id here:
record_set_id = primary_rs_id # e.g. the main record set @id from above
df = dfs[record_set_id]

# Inspect columns for @id-based references
print(f"Columns in DataFrame for {record_set_id}:")
print(df.columns.tolist())

# Suppose one of the numeric field/column @id is 'Age' or similar (replace with actual @id from previous cell if necessary)
numeric_field_id = None
# Try to guess which column represents a numeric field (e.g. contains 'age', 'interval', 'years', etc)
for col in df.columns:
    if 'age' in col.lower() or 'interval' in col.lower() or 'year' in col.lower():
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Just use first numeric-like column (float or int)
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
if numeric_field_id is None:
    print("No numeric field found for filtering. Please adjust numeric_field_id variable.")
else:
    print(f"Using numeric field @id: {numeric_field_id}")

    # Set a threshold, e.g. 50; adjust according to your field
    threshold = 50
    filtered_df = df[df[numeric_field_id] > threshold]

    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head(3))

    # Normalization
    filtered_df = filtered_df.copy()
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head(3))

    # Try to find a group-able field (categorical)
    group_field_id = None
    for col in df.columns:
        # Look for 'sex', 'site', 'type', 'status', 'distribution', or any object/string col
        vals = df[col].dropna().unique()
        if df[col].dtype == object and 1 < len(vals) < min(10, len(df)/2):
            group_field_id = col
            break
    if group_field_id:
        print(f"\nGrouping by field @id: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(grouped_df.head())
    else:
        print("No suitable group field found; please set group_field_id explicitly if needed.")

## 5. Visualization
Visualize the distributions or relationships identified in the data. We'll plot a histogram of the numeric variable and, if a grouping field exists, compare groups with a boxplot.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

if 'group_field_id' in locals() and group_field_id:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.ylabel(numeric_field_id)
    plt.xlabel(group_field_id)
    plt.show()

## 6. Conclusion
This notebook demonstrated step-by-step loading, exploration, and visualization of a Croissant-enabled clinical oncology dataset using the `mlcroissant` library. All references to record sets, fields, and columns were handled using their `@id`, as recommended for reproducible, schema-driven data science workflows.

**Next steps:**
- Deepen EDA or model development using the loaded DataFrames.
- Map `@id` columns to their clinical meaning using accompanying documentation.
- Apply additional filtering, visualization, or ML workflows tailored to your research questions.

For more information about the FAIRˆ2 dataset, see its [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) or visit [SenScience](https://sen.science/).